# 04. Classification & Forecasting Models

## Superstore Sales Data Mining Project

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.features.builder import FeatureBuilder
from src.mining.clustering import ClusterMiner
from src.models.supervised import SupervisedModel
from src.models.forecasting import TimeSeriesModel
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Classification: Predict Customer Segment

In [2]:
# Load data and create features
loader = DataLoader()
df = loader.generate_sample_data(n_orders=2000)

cleaner = DataCleaner(df)
df = cleaner.handle_missing_values()

builder = FeatureBuilder(df)
rfm = builder.create_rfm_features()

# Cluster for labels
feature_cols = ['Recency', 'Frequency', 'Monetary']
scaler = StandardScaler()
features_scaled = scaler.fit_transform(rfm[feature_cols])

clusterer = ClusterMiner(n_clusters=4, random_state=42)
labels = clusterer.fit_kmeans(features_scaled, n_clusters=4)

rfm['Cluster'] = labels
rfm['Segment'] = rfm['Cluster'].map({0: 'Segment A', 1: 'Segment B', 2: 'Segment C', 3: 'Segment D'})

# Create features for classification
rfm['Avg_Profit'] = rfm['Total_Profit'] / rfm['Frequency']
rfm['Avg_Order_Value'] = rfm['Monetary'] / rfm['Frequency']
rfm['Profit_Margin'] = rfm['Total_Profit'] / rfm['Monetary']

X = rfm[['Recency', 'Frequency', 'Monetary', 'Total_Profit', 'Avg_Profit', 'Avg_Order_Value', 'Profit_Margin']]
y = rfm['Segment']

print(f"Features shape: {X.shape}")
print(f"Classes: {y.unique()}")

INFO:src.data.loader:Generated 4982 sample records
INFO:src.data.cleaner:Handling missing values...
INFO:src.features.builder:Creating RFM features...


TypeError: unsupported operand type(s) for +: 'int' and 'datetime.timedelta'

In [ ]:
# Train classification models
clf = SupervisedModel(random_state=42, test_size=0.2)
clf.prepare_data(X, y, scale=True)

# Baseline 1: Logistic Regression
clf.train_baseline_logistic_regression()

# Baseline 2: Decision Tree
clf.train_baseline_decision_tree(max_depth=10)

# Improved: Random Forest
clf.train_improved_random_forest(n_estimators=100, max_depth=15)

In [ ]:
# Model comparison
print("\n=== Classification Model Comparison ===")
comparison = clf.compare_models()
print(comparison)

In [ ]:
# Feature importance (Random Forest)
print("\nFeature Importance (Random Forest):")
importance = clf.get_feature_importance('RandomForest')
print(importance)

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance, x='Importance', y='Feature', palette='viridis', ax=ax)
ax.set_title('Feature Importance - Random Forest')
plt.tight_layout()
plt.savefig('../outputs/figures/10_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/figures/10_feature_importance.png")

In [ ]:
# Confusion matrix for best model
best_model = comparison.iloc[0]['Model']
clf.plot_confusion_matrix(best_model, save_path='../outputs/figures/11_confusion_matrix.png')
print(f"Saved: outputs/figures/11_confusion_matrix.png")

## 2. Forecasting: Sales Prediction

In [ ]:
# Prepare time series
df['Order Date'] = pd.to_datetime(df['Order Date'])

ts_model = TimeSeriesModel(freq='W', test_size=0.2)
ts = ts_model.prepare_time_series(df, 'Order Date', 'Sales', 'sum')
train, test = ts_model.train_test_split(n_test=12)

print(f"Train: {len(train)}, Test: {len(test)}")

In [ ]:
# Baselines
ts_model.baseline_naive()
ts_model.baseline_moving_average(window=4)

# Models
ts_model.fit_arima(order=(1, 1, 1))
ts_model.fit_holt_winters(seasonal='add', seasonal_periods=12)

In [ ]:
# Forecasting comparison
print("\n=== Forecasting Model Comparison ===")
ts_comparison = ts_model.compare_models()
print(ts_comparison)

In [ ]:
# Plot forecast
ts_model.plot_forecast(save_path='../outputs/figures/12_forecast.png')
print("Saved: outputs/figures/12_forecast.png")

## 3. Save Results

In [ ]:
# Save model results
comparison.to_csv('../outputs/tables/04_classification_comparison.csv', index=False)
ts_comparison.to_csv('../outputs/tables/04_forecasting_comparison.csv', index=False)
importance.to_csv('../outputs/tables/04_feature_importance.csv', index=False)

print("Modeling completed!")
print("Saved:")
print("  - outputs/tables/04_classification_comparison.csv")
print("  - outputs/tables/04_forecasting_comparison.csv")
print("  - outputs/tables/04_feature_importance.csv")